# 01 · Data cleaning
**Project:** Why are customers returning orders? Returns, discount & delivery analysis

**Data:** UCI *Online Retail II*, real invoice lines from a UK online gift retailer, 01-Dec-2009 to 09-Dec-2011
(run `python scripts/download_data.py` first). Most customers are small shops and wholesalers.

What this notebook does, step by step:

| Step | What happens |
|---|---|
| 1 | Load the raw file and look at it |
| 2 | Standardise product codes and remove duplicate rows |
| 3 | Label each line: **Sale** or **Return** (credit-note invoices start with `C`) |
| 4 | Split out non-product lines: postage (kept as delivery charges), fees, manual adjustments |
| 5 | Remove stock write-offs and zero-price lines |
| 6 | Remove two bulk-order mistakes |
| 7 | Tidy customer ids, dates and countries |
| 8 | Build the product list and give every product a category |
| 9 | Check the result and save the clean tables to `data/processed/` |

Every rule is written to a cleaning log, so each number can be traced back.

In [1]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
REF = ROOT / "data" / "reference"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

cleaning_log = []

def log(step, rows_before, rows_after, note=""):
    cleaning_log.append({"step": step, "rows_before": rows_before, "rows_after": rows_after,
                         "rows_removed": rows_before - rows_after, "note": note})

## 1. Load and look at the raw data

In [2]:
raw = pd.read_csv(RAW / "online_retail_II.csv", dtype={"Invoice": str, "StockCode": str})
print(f"{len(raw):,} rows, {raw.shape[1]} columns")
raw.head()

1,067,371 rows, 8 columns


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB


In [4]:
# missing values per column
raw.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

**First look:**
- `Customer ID` is missing on about 23% of rows (guest checkouts). These rows are still real sales, so I keep them as customer `Unknown`.
- `Description` is missing on some rows, and these turn out to be stock adjustments with a price of 0.
- `Quantity` is negative on returns and on stock adjustments.
- Column names have spaces and capitals, so I rename them to snake_case.

In [5]:
df = raw.rename(columns={"Invoice": "invoice_no", "StockCode": "stock_code", "Description": "description",
                         "Quantity": "quantity", "InvoiceDate": "invoice_datetime", "Price": "unit_price",
                         "Customer ID": "customer_id", "Country": "country"})
df.columns.tolist()

['invoice_no',
 'stock_code',
 'description',
 'quantity',
 'invoice_datetime',
 'unit_price',
 'customer_id',
 'country']

## 2. Standardise product codes, then remove duplicates
The same product sometimes appears with an upper- and a lower-case code (`15056BL` and `15056bl`), and a few codes have spaces.
Power BI and most databases treat these as one key, so I make every code upper case and trim it **before** looking for duplicates.

In [6]:
before = df["stock_code"].nunique()
df["stock_code"] = df["stock_code"].str.strip().str.upper()
print(f"distinct stock codes: {before:,} -> {df['stock_code'].nunique():,}")

distinct stock codes: 5,305 -> 5,131


In [7]:
n0 = len(df)
df = df.drop_duplicates()
log("Remove exact duplicate rows", n0, len(df))
print(f"duplicates removed: {n0 - len(df):,}")

duplicates removed: 34,335


In [8]:
# a few more lines are the same invoice, product, quantity and price, entered twice a minute apart
# or with a slightly different description: treat these as double entries too
n0 = len(df)
df = df.drop_duplicates(subset=["invoice_no", "stock_code", "quantity", "unit_price", "customer_id", "country"])
log("Remove double entries (same invoice, product, quantity, price)", n0, len(df))
print(f"double entries removed: {n0 - len(df):,}")

double entries removed: 4


## 3. Sale or return?
In this system a return (or a cancellation) is booked as a **credit note**: the invoice number starts with `C` and the quantity is negative.
Invoices starting with `A` are accounting adjustments for bad debt, not customer orders.

In [9]:
df["line_type"] = "Sale"
df.loc[df["invoice_no"].str.startswith("C"), "line_type"] = "Return"
df.loc[df["invoice_no"].str.startswith("A"), "line_type"] = "Adjustment"
df["line_type"].value_counts()

line_type
Sale          1013922
Return          19104
Adjustment          6
Name: count, dtype: int64

In [10]:
# sanity check: returns should always have a negative quantity
df.groupby("line_type")["quantity"].agg(["min", "max"])

,min,max
line_type,,
Adjustment,1,1
Return,-80995,1
Sale,-9600,80995


## 4. Product lines vs. other lines
Real product codes start with 5 digits (e.g. `85123A`). Other codes are not products:

- `POST`, `DOT`, `C2`: postage and carriage charged to the customer. I keep these separately as **delivery charges**.
- `M` (manual), `D` (discount), `BANK CHARGES`, `AMAZONFEE`, `ADJUST`, `TEST`, gift vouchers, samples: fees and manual corrections, which I remove.

In [11]:
is_product = df["stock_code"].str.match(r"^\d{5}")
postage_codes = ["POST", "DOT", "C2"]
is_postage = df["stock_code"].isin(postage_codes)

# what are the non-product codes worth?
df[~is_product].groupby("stock_code").agg(lines=("invoice_no", "size"),
                                          value=("unit_price", lambda p: (p * df.loc[p.index, "quantity"]).sum())) \
               .sort_values("lines", ascending=False).head(12)

,lines,value
stock_code,,
POST,2086,"110,430.41"
DOT,1425,"309,844.10"
M,1392,"-83,311.28"
C2,277,"13,136.00"
D,173,"-12,879.63"
S,101,"-5,991.11"
BANK CHARGES,100,"-35,482.25"
ADJUST,67,"6,835.24"
AMAZONFEE,36,"-221,520.50"


In [12]:
delivery = df[is_postage & (df["line_type"] != "Adjustment")].copy()
delivery["charge_type"] = delivery["stock_code"].map({"POST": "Postage", "DOT": "Dotcom postage", "C2": "Carriage"})

n0 = len(df)
df = df[is_product & (df["line_type"] != "Adjustment")].copy()
log("Keep product lines only (postage saved separately, fees/adjustments removed)", n0, len(df),
    f"{len(delivery):,} postage/carriage lines kept in delivery_charges.csv")
print(f"product lines: {len(df):,} | delivery charge lines: {len(delivery):,}")

product lines: 1,027,052 | delivery charge lines: 3,788


## 5. Stock write-offs and zero-price lines
Some lines have a price of 0. Many have negative quantities and notes such as *damaged*, *lost* or *thrown away*.
These are **warehouse stock adjustments**, not customer sales or returns, so they are removed.
I count them first, because damaged stock is a supply-chain number worth knowing.

In [13]:
zero_price = df["unit_price"] <= 0
writeoffs = df[zero_price & (df["quantity"] < 0)]
print(f"zero-price lines: {zero_price.sum():,} (of which stock write-offs with negative quantity: {len(writeoffs):,})")
writeoffs["description"].str.lower().value_counts().head(10)

zero-price lines: 5,928 (of which stock write-offs with negative quantity: 3,362)


description
check                     124
damaged                    96
damages                    88
?                          81
missing                    30
sold as set on dotcom      20
dotcom                     11
smashed                     9
thrown away                 9
unsaleable, destroyed.      9
Name: count, dtype: int64

In [14]:
n0 = len(df)
df = df[~zero_price]
log("Remove zero-price lines (stock adjustments, free items)", n0, len(df),
    f"{len(writeoffs):,} were stock write-offs (damaged/lost/missing)")

## 6. Two bulk-order mistakes
Two single orders were for 80,995 and 74,215 units and were each reversed by a credit note within 20 minutes.
These were order-entry mistakes, not real returns. Left in, these two lines alone would be about a third of all returned value.

In [15]:
big = df["quantity"].abs() > 50000
df[big][["invoice_no", "line_type", "description", "quantity", "unit_price", "invoice_datetime", "customer_id"]]

,invoice_no,line_type,description,quantity,unit_price,invoice_datetime,customer_id
587080,541431,Sale,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,2011-01-18 10:01:00,"12,346.00"
587085,C541433,Return,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,2011-01-18 10:17:00,"12,346.00"
1065882,581483,Sale,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,2011-12-09 09:15:00,"16,446.00"
1065883,C581484,Return,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,2011-12-09 09:27:00,"16,446.00"


In [16]:
n0 = len(df)
df = df[~big]
log("Remove two bulk-order mistakes (order + same-day reversal)", n0, len(df))

## 7. Customers, dates, countries

In [17]:
# customer id is read as a decimal (12346.0) because of the blanks, so turn it into text, blanks become 'Unknown'
def clean_customer(series):
    return series.map(lambda x: "Unknown" if pd.isna(x) else str(int(x)))

df["customer_id"] = clean_customer(df["customer_id"])
delivery["customer_id"] = clean_customer(delivery["customer_id"])
print(f"lines without a customer id: {(df['customer_id'] == 'Unknown').mean():.1%}")

lines without a customer id: 22.2%


In [18]:
df["invoice_datetime"] = pd.to_datetime(df["invoice_datetime"])
df["invoice_date"] = df["invoice_datetime"].dt.date
delivery["invoice_date"] = pd.to_datetime(delivery["invoice_datetime"]).dt.date
print("date range:", df["invoice_date"].min(), "to", df["invoice_date"].max())

date range: 2009-12-01 to 2011-12-09


In [19]:
# country names are already consistent; map each to a region with a small lookup table
regions = pd.read_csv(REF / "country_regions.csv")
missing = set(df["country"]) - set(regions["country"])
print("countries without a region:", missing or "none")
regions.head()

countries without a region: none


,country,country_clean,region,shipping_zone
0,United Kingdom,United Kingdom,United Kingdom,Domestic
1,Channel Islands,Channel Islands,United Kingdom,Domestic
2,EIRE,Ireland,EU,International
3,Austria,Austria,EU,International
4,Belgium,Belgium,EU,International


In [20]:
df["line_value"] = (df["quantity"] * df["unit_price"]).round(2)
delivery["line_value"] = (delivery["quantity"] * delivery["unit_price"]).round(2)

## 8. Product list and categories
The same stock code sometimes has slightly different descriptions, so I use the most common one.

The data has **no product category**, so I made one: each product gets the category of the first keyword in its description that
matches `data/reference/category_keywords.csv` (e.g. `MUG` → *Kitchen & Dining*, `CANDLE` → *Candles & Lighting*).
This is my own grouping, not the retailer's. Anything that doesn't match goes to *Other*.

In [21]:
desc_counts = (df.dropna(subset=["description"])
                 .assign(description=lambda d: d["description"].str.upper().str.strip())
                 .groupby(["stock_code", "description"]).size().reset_index(name="lines")
                 .sort_values("lines", ascending=False))
products = desc_counts.drop_duplicates("stock_code")[["stock_code", "description"]]

keywords = pd.read_csv(REF / "category_keywords.csv").sort_values("priority", kind="stable")

def categorise(description):
    words = set(re.split(r"[^A-Z0-9'\-]+", description))
    for keyword, category in zip(keywords["keyword"], keywords["category"]):
        if keyword in words:
            return category
    return "Other"

products["category"] = products["description"].map(categorise)
products = products.sort_values("stock_code").reset_index(drop=True)
products["category"].value_counts()

category
Home Decor & Textiles      941
Kitchen & Dining           891
Jewellery & Accessories    522
Other                      491
Stationery & Gift Wrap     437
Candles & Lighting         423
Bags & Storage             324
Christmas & Seasonal       282
Toys & Party               247
Garden & Outdoor           162
Name: count, dtype: int64

In [22]:
# every product line must have a product in the product list
assert df["stock_code"].isin(products["stock_code"]).all()
products.sample(8, random_state=1)

,stock_code,description,category
3239,72128,COLUMBIAN CANDLE ROUND,Candles & Lighting
2954,35980B,RED TOP SCANDINAVIAN HEART,Home Decor & Textiles
3370,79067,CORONA MEXICAN TRAY,Kitchen & Dining
2949,35972,DAISY FOLKART HEART DECORATION,Home Decor & Textiles
3503,84206B,CAT W SUNGLASSES BLANK CARD,Stationery & Gift Wrap
3607,84510C,SET OF 4 POLKADOT COASTERS,Kitchen & Dining
4002,84971S,SMALL HEART FLOWERS HOOK,Home Decor & Textiles
3100,47348A,FUSCHIA VOILE POINTY SHOE DEC,Other


## 9. Customers table, checks and export

In [23]:
known = df[df["customer_id"] != "Unknown"]
customers = known.groupby("customer_id").agg(country=("country", lambda c: c.mode()[0])).reset_index()

# first purchase = first SALE line (some customers only appear with a return of something bought before Dec-2009)
first_sale = known[known["line_type"] == "Sale"].groupby("customer_id")["invoice_date"].min()
customers["first_purchase_date"] = customers["customer_id"].map(first_sale)
print(f"{len(customers):,} known customers, {customers['first_purchase_date'].isna().sum()} with returns only")
customers.head()

5,875 known customers, 23 with returns only


,customer_id,country,first_purchase_date
0,12346,United Kingdom,2010-03-02
1,12347,Iceland,2010-10-31
2,12348,Finland,2010-09-27
3,12349,Italy,2010-04-29
4,12350,Norway,2011-02-02


In [24]:
checks = {
    "no exact duplicates left": not df.duplicated().any(),
    "returns all have negative quantity": (df.loc[df.line_type == "Return", "quantity"] < 0).all(),
    "sales all have positive quantity": (df.loc[df.line_type == "Sale", "quantity"] > 0).all(),
    "all prices > 0": (df["unit_price"] > 0).all(),
    "every country has a region": df["country"].isin(regions["country"]).all(),
    "every product has a category": products["category"].notna().all(),
    "customer ids unique in customers table": customers["customer_id"].is_unique,
}
pd.Series(checks, name="passed").to_frame()

,passed
no exact duplicates left,True
returns all have negative quantity,True
sales all have positive quantity,True
all prices > 0,True
every country has a region,True
every product has a category,True
customer ids unique in customers table,True


In [25]:
assert all(checks.values())

order_lines = df[["invoice_no", "invoice_date", "line_type", "stock_code", "quantity", "unit_price",
                  "line_value", "customer_id", "country"]].sort_values(["invoice_date", "invoice_no"])
delivery_out = delivery[["invoice_no", "invoice_date", "line_type", "charge_type", "line_value", "customer_id", "country"]]

order_lines.to_csv(OUT / "order_lines.csv", index=False)
products.to_csv(OUT / "products.csv", index=False)
customers.to_csv(OUT / "customers.csv", index=False)
regions.to_csv(OUT / "countries.csv", index=False)
delivery_out.to_csv(OUT / "delivery_charges.csv", index=False)

for name, t in [("order_lines", order_lines), ("products", products), ("customers", customers),
                ("countries", regions), ("delivery_charges", delivery_out)]:
    print(f"{name:<17} {len(t):>9,} rows")

order_lines       1,021,120 rows
products              4,720 rows
customers             5,875 rows
countries                43 rows
delivery_charges      3,788 rows


In [26]:
log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(OUT / "cleaning_log.csv", index=False)
log_df

,step,rows_before,rows_after,rows_removed,note
0,Remove exact duplicate rows,1067371,1033036,34335,
1,"Remove double entries (same invoice, product, ...",1033036,1033032,4,
2,Keep product lines only (postage saved separat...,1033032,1027052,5980,"3,788 postage/carriage lines kept in delivery_..."
3,"Remove zero-price lines (stock adjustments, fr...",1027052,1021124,5928,"3,362 were stock write-offs (damaged/lost/miss..."
4,Remove two bulk-order mistakes (order + same-d...,1021124,1021120,4,


In [27]:
# quick summary of what is left
summary = order_lines.groupby("line_type").agg(lines=("invoice_no", "size"), invoices=("invoice_no", "nunique"),
                                               value=("line_value", "sum"))
summary

,lines,invoices,value
line_type,,,
Return,17912,7403,"-470,772.77"
Sale,1003208,39514,"19,396,453.50"


### Key cleaning decisions (the ones to explain in an interview)
- **Returns = credit notes.** Lines on invoices starting with `C`. The data doesn't separate "cancelled before dispatch" from
  "sent back after delivery", so "returns" here means both.
- **Postage lines are kept, not deleted.** They are the only delivery information in the data, so they go to `delivery_charges.csv`.
- **Stock write-offs are removed.** Zero-price lines marked *damaged / lost / missing* are warehouse adjustments, not customer returns.
- **Two bulk-order mistakes removed.** Each was reversed within 20 minutes and would distort every return number.
- **Missing customer ids kept as `Unknown`.** Dropping them would remove about a fifth of real sales. They are only left out of customer-level analysis.
- **Categories are my own keyword grouping**, because the retailer's data has none.

Next: `sql/` (analysis) and `02_returns_analysis.ipynb` (charts).